# 🎵 Обучение Custom Demucs на MUSDB18 — полный режим

Ноутбук сам определяет, какой датасет доступен, готовит его и обучает модель.

| Режим | Что это | Качество |
|---|---|---|
| **full** | полный MUSDB18: 100 треков по ~4 минуты, лежит на Google Drive | настоящее — в 30 раз больше звука, чем в sample |
| **sample** | 7-секундные выдержки MUSDB18: 100 треков по 7 секунд | демо — общий принцип поймёт, для продакшена мало |

**Что происходит:** поиск данных → конвертация в формат проекта → (для маленьких
данных) лёгкое разнообразивание → 30 эпох обучения → сохранение модели на Drive.

## Про время и «неожиданную» скорость

30 эпох на Tesla T4 — ориентировочно **15–25 минут**. Да, быстро, и вот почему:
за одну эпоху сеть берёт **один случайный кусок 7.8 секунды из каждого трека**,
а не всю песню целиком. Полная 4-минутная дорожка и 7-секундная обрезка стоят
в эпоху одинаково. Обратная сторона: за эпоху сеть видит лишь малую часть
каждой длинной дорожки, поэтому 30 эпох — это «быстрый прогон», а не полная
проработка датасета.

⚠️ **Не жми «Run all» вслепую.** Сначала дождись, пока конвертация напечатает
число найденных треков, и убедись, что оно не ноль.

In [ ]:
# Подключаем Google Drive — оттуда берём полный MUSDB18 и туда же сохраняем модель.
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# Клонируем репозиторий с моделью и обучающим кодом.
import os
import shutil

REPO = "https://github.com/IgorCim/demucs_custom.git"
PROJECT_DIR = "/content/demucs_custom"

if os.path.isdir(PROJECT_DIR):
    print("Репозиторий уже скачан, обновляю.")
    shutil.rmtree(PROJECT_DIR)

!git clone --depth 1 {REPO}
os.chdir(PROJECT_DIR)
print("Работаем в:", os.getcwd())

In [ ]:
# Зависимости. demucs ставится из PyPI, но локальная папка репозитория
# перекрывает его — используется наша модифицированная версия.
#
# musdb ставится отдельно: в requirements.txt его нет, а без него
# не скачаются 7-секундные выдержки.
!pip install -q -r requirements.txt
!pip install -q musdb

# sanity check
import importlib
for name in ("torch", "numpy", "soundfile", "librosa", "musdb", "demucs"):
    importlib.import_module(name)
print("Все зависимости на месте.")

In [ ]:
# Проверяем GPU до обучения — на CPU 30 эпох считать бессмысленно.
import torch

print("torch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Видеопамять: {total_gb:.1f} ГБ")
else:
    raise RuntimeError(
        "GPU не найдена. В Colab: Среда выполнения -> Сменить среду выполнения "
        "-> Аппаратный ускоритель -> GPU (T4)."
    )

In [ ]:
# Все пути и константы обучения.
#
# ВАЖНО про длину куска: 343980 отсчётов = 7.8 секунды. Это ЖЁСТКИЙ ПРЕДЕЛ
# pretrained htdemucs, а не наш выбор. Его segment = 39/5 секунды, и сеть
# отказывается обрабатывать что-то длиннее. Проверено на demucs 4.1.0:
#   343980 -> ок
#   480000 -> RuntimeError: shape '[1, 4, -1, 343980]' is invalid
#   485100 (44100*11) -> ValueError про training length
# Значение 480000 (10.88 сек) — это длина сегмента в НЕ-hybrid demucs,
# у htdemucs она своя. Поэтому здесь НЕ 11 секунд.

SEGMENT_LENGTH = 343980          # отсчётов, 7.8 сек — предел htdemucs
TARGET_SR = 44100                # частота дискретизации
EPOCHS = 30
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
VALID_RATIO = 0.1                # 10% треков уйдёт в валидацию

# Готовим данные на ЛОКАЛЬНОМ диске Colab — не на Drive.
# С Drive идёт постоянное чтение, оно убивает скорость обучения в разы.
# С Drive мы только ЧИТАЕМ исходный датасет (один раз) и ПИШЕМ готовую модель.
CONVERTED_ROOT = "/content/custom_dataset_full"

OUTPUT_MODEL_PATH = (
    "/content/drive/MyDrive/demucs_custom/models/custom_demucs_final.pth"
)

print(f"Кусок:      {SEGMENT_LENGTH} отсчётов = {SEGMENT_LENGTH / TARGET_SR:.1f} сек")
print(f"Данные:     {CONVERTED_ROOT}")
print(f"Модель:     {OUTPUT_MODEL_PATH}")

In [ ]:
# Ячейка 7 — автоопределение источника данных.
#
# Приоритет:
#   1. полный MUSDB18 (больше 50 треков в train) на Drive или на локальном диске
#   2. 7-секундные выдержки MUSDB18
#   3. автоматическая загрузка выдержек
import os

SOURCES = ["vocals", "drums", "bass", "other"]


def find_best_dataset():
    full_musdb_paths = [
        '/content/drive/MyDrive/demucs_custom/data/musdb18_full',
        '/content/musdb18_full',
    ]

    for path in full_musdb_paths:
        train = os.path.join(path, "train")
        # больше 50 треков = это точно полный датасет, а не выдержки
        if os.path.isdir(train) and len(os.listdir(train)) > 50:
            print(f"✅ Полный MUSDB18: {path}")
            return path, 'full'

    sample_path = '/content/musdb18'
    if os.path.isdir(os.path.join(sample_path, "train")):
        print(f"⚠️  7-сек выдержки уже скачаны: {sample_path}")
        return sample_path, 'sample'

    print("⬇️  Скачиваю 7-сек выдержки MUSDB18 (~1 ГБ, 1-2 минуты)...")
    import musdb
    musdb.DB(root=sample_path, download=True)

    # musdb иногда отдаёт пустую папку при обрыве связи — проверяем,
    # иначе обучение молча ушло бы в режим "демо" на пустом месте.
    train = os.path.join(sample_path, "train")
    if not os.path.isdir(train) or not os.listdir(train):
        raise RuntimeError(
            f"Скачалось пусто: {sample_path}\n"
            "Проверь соединение и запусти ячейку заново."
        )

    print(f"✅ Выдержки готовы: {sample_path}")
    return sample_path, 'sample'


data_path, data_type = find_best_dataset()

# Считаем, что реально доступно для обучения.
raw_train_dir = os.path.join(data_path, "train")
n_available = len(os.listdir(raw_train_dir)) if os.path.isdir(raw_train_dir) else 0

if data_type == 'full':
    print(f"\nРежим: FULL   | треков в train: {n_available} | полные песни ~4 мин")
    print("Ожидаем: 15-25 минут на 30 эпох, качество — настоящее.")
else:
    print(f"\nРежим: SAMPLE | треков в train: {n_available} | по 7 секунд")
    print("Ожидаем: 10-15 минут на 30 эпох, качество — демонстрационное.")
    print("Для настоящего качества загрузи полный MUSDB18 (инструкция в конце).")

In [ ]:
# Ячейка 8 — конвертация в формат проекта.
#
# ЭТО САМЫЙ ВАЖНЫЙ ШАГ: без него обучение не запустится.
#
# MUSDB лежит так:
#     musdb18/train/Track Name/vocals.wav
#                            drums.wav
#                            bass.wav
#                            other.wav
#     (смеси mixture.wav там НЕТ, и папки valid тоже нет)
#
# А проект ждёт так:
#     custom_dataset_full/train/track_001/{mixture,vocals,drums,bass,other}.wav
#     custom_dataset_full/valid/track_091/...
#
# Что делаем:
#   * собираем mixture как сумму четырёх дорожек
#   * отбираем только годные треки
#   * режем на train/valid уже ПОСЛЕ отбора, иначе валидация может остаться пустой
#   * пишем 16-битный PCM — вдвое меньше места, чем float32
#
# ВАЖНО: используем только папку train. В MUSDB есть ещё test (50 треков) —
# это официальный отладочный набор, на нём нельзя обучать.
import time

import numpy as np
import soundfile as sf

if os.path.isdir(os.path.join(CONVERTED_ROOT, "train")) and os.path.isdir(
    os.path.join(CONVERTED_ROOT, "valid")
):
    print("✅ Датасет уже сконвертирован, пропускаем.")
else:
    os.makedirs(CONVERTED_ROOT, exist_ok=True)
    for split in ("train", "valid"):
        os.makedirs(os.path.join(CONVERTED_ROOT, split), exist_ok=True)

    def find_stem(track_dir, name):
        """Ищет дорожку: .wav, .stem.wav или .flac — MUSDB встречается по-разному."""
        for candidate in (f"{name}.wav", f"{name}.stem.wav", f"{name}.flac"):
            path = os.path.join(track_dir, candidate)
            if os.path.isfile(path):
                return path
        return None

    def read_stereo(path, max_samples):
        """Читает файл, приводит к 2 каналам, режет по длине."""
        audio, sr = sf.read(path, dtype="float32", always_2d=True)
        if audio.shape[1] == 1:
            audio = np.repeat(audio, 2, axis=1)       # моно -> стерео
        elif audio.shape[1] > 2:
            audio = audio[:, :2]                      # лишние каналы отбрасываем
        if sr != TARGET_SR:
            import librosa
            audio = librosa.resample(
                audio.T, orig_sr=sr, target_sr=TARGET_SR
            ).T
        if audio.shape[1] > max_samples:
            audio = audio[:, :max_samples]
        return np.ascontiguousarray(audio.T, dtype="float32")   # -> (2, T)

    def write_wav(path, audio, gain=1.0):
        """16-битный PCM: вдвое меньше места, чем float32.

        gain применяется ОДИН И ТОТ ЖЕ ко всем пяти файлам трека.
        Нормировать каждый файл отдельно нельзя: сумма четырёх дорожек
        почти всегда выходит за 1.0, и mixture тогда уехал бы вниз
        относительно дорожек (проверено: расхождение до 0.73).
        """
        sf.write(path, (audio * gain).T, TARGET_SR, subtype="PCM_16")

    MAX_SAMPLES = 10 * 60 * TARGET_SR      # не держим трек дольше 10 минут
    MIN_SAMPLES = 10 * TARGET_SR           # короче 10 секунд — мусор, пропускаем

    # --- Фаза 1: отбор годных треков -------------------------------------
    raw_tracks = sorted(
        os.path.join(raw_train_dir, d)
        for d in os.listdir(raw_train_dir)
        if os.path.isdir(os.path.join(raw_train_dir, d))
    )
    print(f"В папке train найдено папок: {len(raw_tracks)}")

    usable = []
    for track_dir in raw_tracks:
        stems = {}
        for name in SOURCES:
            path = find_stem(track_dir, name)
            if path is None:
                break
            stems[name] = path
        if len(stems) != len(SOURCES):
            missing = [n for n in SOURCES if n not in stems]
            print(f"  ⚠️  пропускаю {os.path.basename(track_dir)}: нет {missing}")
            continue
        try:
            frames = min(sf.info(stems[n]).frames for n in SOURCES)
        except Exception as exc:
            print(f"  ⚠️  не читается {os.path.basename(track_dir)}: {exc}")
            continue
        if frames < MIN_SAMPLES:
            print(f"  ⚠️  пропускаю {os.path.basename(track_dir)}: слишком короткий")
            continue
        usable.append(stems)

    if not usable:
        raise RuntimeError(
            "Ни один трек не прошёл проверку. Проверь, что в "
            f"{raw_train_dir} внутри каждой папки лежат 4 дорожки."
        )

    # --- Фаза 2: делим на train/valid -------------------------------------
    n_valid = max(1, int(len(usable) * VALID_RATIO))
    print(f"\nГодных треков: {len(usable)} → train {len(usable) - n_valid}, valid {n_valid}")

    # --- Фаза 3: конвертируем ---------------------------------------------
    started = time.time()
    written = 0
    for index, stems in enumerate(usable):
        try:
            audio = {name: read_stereo(stems[name], MAX_SAMPLES) for name in SOURCES}
        except Exception as exc:
            print(f"  ⚠️  не читается трек #{index + 1}: {exc}")
            continue

        length = min(min(a.shape[1] for a in audio.values()), MAX_SAMPLES)
        if length < MIN_SAMPLES:
            print(f"  ⚠️  пропускаю трек #{index + 1}: слишком короткий")
            continue

        split = "valid" if index >= len(usable) - n_valid else "train"
        out_dir = os.path.join(CONVERTED_ROOT, split, f"track_{index + 1:03d}")
        os.makedirs(out_dir, exist_ok=True)

        mixture = np.zeros((2, length), dtype="float32")
        for name in SOURCES:
            mixture += audio[name][:, :length]

        # Один общий коэффициент на весь трек: пик суммы четырёх дорожек
        # обычно больше 1.0, и без этого mixture не равна сумме дорожек.
        peak = float(np.max(np.abs(mixture)))
        gain = (1.0 / peak) if peak > 1.0 else 1.0
        if gain != 1.0:
            print(f"  · трек #{index + 1}: пик {peak:.2f} -> gain {gain:.3f}")

        for name in SOURCES:
            chunk = np.ascontiguousarray(audio[name][:, :length])
            write_wav(os.path.join(out_dir, f"{name}.wav"), chunk, gain)
        write_wav(os.path.join(out_dir, "mixture.wav"), mixture, gain)

        written += 1
        if written % 20 == 0:
            print(f"  ... готово {written} из {len(usable)}")

    print(f"\n✅ Конвертация закончена за {time.time() - started:.0f} сек: {written} треков")
    print(f"   Папка: {CONVERTED_ROOT}")

    # --- Проверка, без которой обучение ушло бы в тихий 2-эпоховый режим ---
    made = [
        d
        for d in os.listdir(os.path.join(CONVERTED_ROOT, "train"))
        if os.path.isfile(os.path.join(CONVERTED_ROOT, "train", d, "mixture.wav"))
    ]
    if not made:
        raise RuntimeError(
            "Ни в одном треке нет mixture.wav — обучение ушло бы в "
            "демо-режим на 2 эпохи. Конвертация сломана."
        )
    print(f"   Проверка: mixture.wav есть в {len(made)} треках — обучение полноценное.")

In [ ]:
# Ячейка 9 — разнообразивание данных.
#
# Честно о том, что это даёт: НИЧЕГО нового по содержанию. Мы не добываем
# ни одной секунды нового звука — делаем вариации уже имеющихся 7-секундных
# обрезков. Это может немного помочь не переобучиться под ровно эти куски,
# но НЕ превращает 100×7 секунд в полноценный датасет. Если хочется качества —
# загружайте полный MUSDB18 (инструкция в конце ноутбука).
#
# Для полного датасета аугментация выключена: там и так 100 длинных дорожек,
# а копии только замедляют обучение.
#
# Что делаем (для каждого трека из train/):
#   * копии со сдвигом тональности — одинаковым для ВСЕХ дорожек трека
#   * лёгкий шум в СМЕСЬ, дорожки-цели остаются чистыми
#
# Почему смесь собирается ИЗ сдвинутых дорожек, а не сдвигается отдельно:
# pitch_shift нелинеен (внутри фазовый вокодер) — проверено: расхождение
# достигает 100% амплитуды. Если сдвинуть дорожки по отдельности, mixture
# перестанет быть их суммой, и обучение станет бессмысленным. Поэтому сначала
# сдвигаем 4 дорожки, затем складываем их и добавляем шум.
#
# valid/ НЕ трогаем: иначе метрика на валидации станет скакать
# и выбор лучшей модели будет ненадёжным.
#
# ВНИМАНИЕ: здесь НЕ используется torchaudio.load/save — в этой версии
# они требуют torchcodec и падают. Читаем и пишем через soundfile.
if data_type == 'full':
    print("Режим FULL — данные достаточно, аугментация не нужна.")
    print("Копии длинных дорожек только замедлили бы обучение.")
else:
    try:
        import librosa
    except ImportError as exc:
        # Датасет к этому моменту уже готов и корректен. Аугментация —
        # приятное дополнение, а не условие обучения. Не роняем прогон.
        print(f"⚠️  librosa не установился ({exc}), пропускаю аугментацию.")
        print("   Обучение запустится на исходных данных — это нормально.")
        librosa = None

    if librosa is not None:
        NUM_COPIES = 3
        SEMITONES = [-2.0, 1.5, 3.0]
        NOISE_LEVEL = 0.002          # тихий, чтобы не выбить цели из смеси

        train_dir = os.path.join(CONVERTED_ROOT, "train")
        tracks = sorted(
            d for d in os.listdir(train_dir)
            if os.path.isdir(os.path.join(train_dir, d))
        )
        print(f"Аугментация: {len(tracks)} треков × {NUM_COPIES} копии")

        rng = np.random.default_rng(42)      # фиксированный seed — повторяемо
        started = time.time()
        created = 0

        for track in tracks:
            src_dir = os.path.join(train_dir, track)

            # Сначала читаем и сдвигаем дорожки. Отдельной "дорожки-эталона"
            # у MUSDB нет — эталон ровно тот, что записан в mixture.wav.
            for copy_index, semitones in enumerate(SEMITONES[:NUM_COPIES]):
                dst_dir = os.path.join(train_dir, f"{track}_aug{copy_index + 1}")
                if os.path.isdir(dst_dir):
                    continue

                shifted = []
                ok = True
                for name in SOURCES:
                    path = os.path.join(src_dir, f"{name}.wav")
                    if not os.path.isfile(path):
                        ok = False
                        break
                    audio, sr = sf.read(path, dtype="float32", always_2d=True)
                    audio = audio.T                       # -> (2, T)
                    if abs(semitones) > 1e-6:
                        audio = np.stack([
                            librosa.effects.pitch_shift(
                                audio[ch], sr=sr, n_steps=semitones
                            )
                            for ch in range(audio.shape[0])
                        ])
                    shifted.append(np.ascontiguousarray(audio, dtype="float32"))

                if not ok:
                    continue

                # Смесь собираем ИЗ СДВИНУТЫХ дорожек, а не сдвигаем отдельно.
                # Иначе mixture перестала бы быть их суммой: pitch_shift
                # нелинеен (фазовый вокодер), и независимый сдвиг дорожек
                # давал бы расхождение до 100% амплитуды. Так эталон и вход
                # гарантированно согласованы.
                stems = np.stack(shifted, axis=0)       # (4, 2, T)
                mixture = stems.sum(axis=0)
                # шум только в смеси: сеть учится доставать чистые цели
                # из слегка грязной смеси, цели остаются чистыми
                mixture = mixture + rng.normal(
                    0.0, NOISE_LEVEL, mixture.shape
                ).astype("float32")

                os.makedirs(dst_dir, exist_ok=True)
                # тот же общий gain на все пять файлов, что и при конвертации
                peak = float(np.max(np.abs(mixture)))
                gain = (1.0 / peak) if peak > 1.0 else 1.0
                for index, name in enumerate(SOURCES):
                    write_wav(os.path.join(dst_dir, f"{name}.wav"), stems[index], gain)
                write_wav(os.path.join(dst_dir, "mixture.wav"), mixture, gain)
                created += 1

        total = len(tracks) + created
        print(f"✅ Создано копий: {created} за {time.time() - started:.0f} сек")
        print(f"   Треков в train теперь: {total}")
        print("   ⚠️  Это вариации тех же 7 секунд. Нового звука не появилось.")

In [ ]:
# Ячейка 10 — обучение.
#
# ВАЖНО: segment_length = 343980, а не 480000 и не 44100*11. Это предел
# pretrained htdemucs. С значением 480000 обучение падает с RuntimeError,
# с 485100 (11 сек) — с ValueError про training length.
import importlib
import sys

for name in ("config", "train_custom", "data_prep", "model_loader", "train_epoch"):
    sys.modules.pop(name, None)

from config import Config
from train_custom import train_custom_model

config = Config()

config.device = "cuda"
config.segment_length = SEGMENT_LENGTH      # 343980 — обязательно
config.batch_size = BATCH_SIZE
config.epochs = EPOCHS
config.learning_rate = LEARNING_RATE
config.custom_data_path = CONVERTED_ROOT
config.output_model_path = OUTPUT_MODEL_PATH

print("Параметры обучения:")
print(f"  режим данных      : {data_type}")
print(f"  устройство        : {config.device}")
print(f"  длина куска       : {config.segment_length} отсчётов "
      f"({config.segment_length / TARGET_SR:.1f} сек)")
print(f"  батч              : {config.batch_size}")
print(f"  эпох              : {config.epochs}")
print(f"  learning rate     : {config.learning_rate}")
print(f"  данные            : {config.custom_data_path}")
print(f"  модель            : {config.output_model_path}")

# Сколько реально будет итераций — чтобы понимать, чего ждать.
try:
    n_train = len([
        d for d in os.listdir(os.path.join(CONVERTED_ROOT, "train"))
        if os.path.isdir(os.path.join(CONVERTED_ROOT, "train", d))
    ])
    iters = (n_train // config.batch_size) * config.epochs
    print(f"  треков в train    : {n_train}")
    print(f"  итераций всего    : ~{iters}")
except Exception:
    pass

# epochs_explicit=True — просим обучение НЕ решать, что 30 эпох это слишком
# много, иначе возможен тихий срез до 2 эпох.
best_model_path = train_custom_model(config, assume_yes=True, epochs_explicit=True)

print("\n" + "=" * 70)
print("✅ ОБУЧЕНИЕ ЗАВЕРШЕНО")
print(f"Модель сохранена: {best_model_path}")
print("=" * 70)

## Как получить полный MUSDB18 (режим `full`)

Только `musdb.DB(download=True)` скачивает **7-секундные выдержки** — это
официальное поведение пакета. Полный набор лежит на Zenodo и требует
одобрения заявки, которую вручную рассматривают (обычно до суток).

### Что сделать

1. **Запроси доступ.** Открой страницу набора на Zenodo и подай заявку,
   согласившись с лицензией CC BY-NC-SA 4.0 на некоммерческое использование.
2. **Скачай** полный архив (около 10 ГБ) и **распакуй** так, чтобы получилось:

   ```
   musdb18_full/
     train/            <- 100 треков, из них ноутбук возьмёт 90 в train и 10 в valid
       Track Name/
         vocals.wav
         drums.wav
         bass.wav
         other.wav
     test/             <- 50 треков, НЕ используются для обучения
   ```

3. **Положи** папку `musdb18_full` в Google Drive сюда:

   ```
   MyDrive/demucs_custom/data/musdb18_full/
   ```

4. **Перезапусти** ноутбук с ячейки 7. Он сам найдёт полный датасет
   (проверяет, что треков больше 50) и переключится в режим `full`.
   Конвертацию делать заново не нужно — она пропустит уже готовую папку,
   так что при смене режима её стоит удалить:
   `!rm -rf /content/custom_dataset_full`

### Почему папка `test` не используется

Это официальный отладочный набор. Если обучиться на нём, метрика
качества перестанет что-либо значить.

## Если что-то пошло не так

| Симптом | Причина | Что делать |
|---|---|---|
| `RuntimeError: shape '[1, 4, -1, 343980]' is invalid` | где-то осталось `segment_length = 480000` | проверить значение в ячейке 10 |
| `ValueError: Given length ... is longer than training length 343980` | задан кусок длиннее 7.8 сек | вернуть `SEGMENT_LENGTH = 343980` |
| `torchcodec` / ошибки `torchaudio.load` | torchaudio 2.11 требует torchcodec | в ноутбуке звук читается через `soundfile`, это обойдено |
| обучение закончилось за 2 эпохи | не нашлось ни одного `mixture.wav` | сработала защита в ячейке 8 — прочитать её сообщение |
| `CUDA available: False` | в Colab не выбрана GPU | Среда выполнения → Аппаратный ускоритель → GPU (T4) |
| Colab отвалился по таймауту | бесплатный Colab обрывает долгие сессии | сохранить чекпойнты на Drive, запускать заново |